# 🗺️ Cartographer — Deep Research Agent

> *"Every great discovery begins with an unexplored question. Cartographer maps the web so you don't have to."*

## Architecture
```
Quest → [Planner] → [Explorer] → [Critic] ──(gaps?)──► [Explorer]
                                           └──(done)──► [Writer] → Treasure Map
```

**Nodes:**
- **Planner** — Decomposes the Quest into 3–5 Waypoints (sub-queries)
- **Explorer** — Parallel Tavily searches per Waypoint
- **Critic** — LLM-as-judge scores coverage (0–10), identifies Uncharted Zones
- **Writer** — Synthesizes the Treasure Map (cited Markdown report)

---

## 0. Setup & Environment

In [ ]:
# Install dependencies (run once)
# !pip install -r requirements.txt

In [ ]:
import sys
from pathlib import Path

# Add project root to path so src/ is importable
project_root = Path().resolve()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from dotenv import load_dotenv
load_dotenv()

print('✅ Environment loaded')

## 1. Verify Configuration

In [ ]:
import os

checks = {
    'GOOGLE_API_KEY': os.getenv('GOOGLE_API_KEY'),
    'TAVILY_API_KEY': os.getenv('TAVILY_API_KEY'),
    'LLM Provider': os.getenv('CARTOGRAPHER_LLM_PROVIDER', 'google'),
    'LLM Model': os.getenv('CARTOGRAPHER_LLM_MODEL', 'gemini-2.0-flash'),
    'Max Expeditions': os.getenv('MAX_EXPEDITIONS', '2'),
    'Critic Threshold': os.getenv('CRITIC_SCORE_THRESHOLD', '7.0'),
}

for k, v in checks.items():
    if v and 'KEY' in k:
        print(f'  ✅ {k}: ***{v[-4:]}')
    elif v:
        print(f'  ✅ {k}: {v}')
    else:
        print(f'  ❌ {k}: NOT SET')

## 2. LLM Factory — Model-Agnostic Setup

In [ ]:
from src.llm_factory import build_llm

# Default: reads from env (CARTOGRAPHER_LLM_PROVIDER + CARTOGRAPHER_LLM_MODEL)
llm = build_llm()
print(f'✅ LLM ready: {llm.__class__.__name__}')

# To switch provider, pass args directly:
# llm = build_llm(provider='anthropic', model='claude-3-haiku-20240307')
# llm = build_llm(provider='openai', model='gpt-4o-mini')

## 3. Individual Node Walkthroughs

Run each node in isolation to understand its behaviour before running the full graph.

### 3a. Planner Node — Chart the Territory

In [ ]:
from src.nodes.planner import planner_node

QUEST = "What are the trade-offs between vector databases for production RAG systems?"

planner_out = await planner_node({'quest': QUEST})

waypoints = planner_out['waypoints']
print(f'📍 Waypoints ({len(waypoints)}):')
for i, w in enumerate(waypoints, 1):
    print(f'  {i}. {w}')

print(f'\n📋 Trace: {planner_out["trace"][0]["message"]}')

### 3b. Explorer Node — Explore the Terrain

In [ ]:
from src.nodes.explorer import explorer_node

explorer_state = {
    'quest': QUEST,
    'waypoints': waypoints,
    'terrain': [],
    'uncharted_zones': [],
    'expedition_count': 0,
}

explorer_out = await explorer_node(explorer_state)
terrain = explorer_out['terrain']

print(f'🔍 Found {len(terrain)} results:')
for r in terrain[:3]:  # Show first 3
    print(f'  [{r["score"]:.2f}] {r["title"]}')
    print(f'        {r["url"]}')
    print(f'        {r["content"][:120]}…\n')

### 3c. Critic Node — Verify the Map

In [ ]:
from src.nodes.critic import critic_node

critic_state = {
    'quest': QUEST,
    'waypoints': waypoints,
    'terrain': terrain,
    'expedition_count': 1,
    'uncharted_zones': [],
    'coverage_score': 0.0,
}

critic_out = await critic_node(critic_state)

print(f'⚖️  Coverage Score: {critic_out["coverage_score"]:.1f}/10')
print(f'   Uncharted Zones: {critic_out["uncharted_zones"]}')
print(f'   Trace: {critic_out["trace"][0]["message"]}')

### 3d. Writer Node — Draw the Map

In [ ]:
from src.nodes.writer import writer_node
from IPython.display import Markdown, display

writer_state = {
    'quest': QUEST,
    'waypoints': waypoints,
    'terrain': terrain,
    'coverage_score': critic_out['coverage_score'],
    'uncharted_zones': critic_out['uncharted_zones'],
    'expedition_count': 1,
    'treasure_map': '',
    'sources': [],
    'trace': [],
}

writer_out = await writer_node(writer_state)

print(f'🗺️ Treasure Map generated ({len(writer_out["treasure_map"])} chars)')
print(f'   Sources cited: {len(writer_out["sources"])}')
print('---')
display(Markdown(writer_out['treasure_map']))

## 4. Full Graph Run

Now run the complete Cartographer graph end-to-end.

In [ ]:
from src.graph import cartographer
from IPython.display import Markdown, display, clear_output
import ipywidgets as widgets

QUEST = "What are the key risks and mitigation strategies for deploying LLMs in healthcare?"

initial_state = {
    'quest': QUEST,
    'waypoints': [],
    'terrain': [],
    'uncharted_zones': [],
    'coverage_score': 0.0,
    'expedition_count': 0,
    'treasure_map': '',
    'sources': [],
    'trace': [],
}

# ── Live streaming output widgets ─────────────────────────────────────────────
log_out = widgets.Output()
map_out = widgets.Output()

header = widgets.HTML('<h3>📍 Expedition Log</h3>')
map_header = widgets.HTML('<h3>📜 Treasure Map (streaming…)</h3>')

display(widgets.HBox([
    widgets.VBox([header, log_out], layout=widgets.Layout(width='40%')),
    widgets.VBox([map_header, map_out], layout=widgets.Layout(width='60%')),
]))

final_state = None

async for event in cartographer.astream_events(initial_state, version='v2'):
    kind = event['event']
    name = event.get('name', '')

    if kind == 'on_chain_end' and name in ('planner', 'explorer', 'critic', 'writer'):
        output = event.get('data', {}).get('output', {})
        for entry in output.get('trace', []):
            with log_out:
                display(Markdown(entry['message']))

    elif kind == 'on_chat_model_stream':
        chunk = event.get('data', {}).get('chunk')
        if chunk and hasattr(chunk, 'content') and chunk.content:
            with map_out:
                print(chunk.content, end='', flush=True)

    elif kind == 'on_chain_end' and name == 'LangGraph':
        final_state = event.get('data', {}).get('output', {})

print('\n✅ Expedition complete!')

In [ ]:
# Final summary stats
if final_state:
    print(f'Coverage Score : {final_state.get("coverage_score", "N/A"):.1f}/10')
    print(f'Expeditions    : {final_state.get("expedition_count", 0)}')
    print(f'Sources cited  : {len(final_state.get("sources", []))}')
    print(f'Report length  : {len(final_state.get("treasure_map", ""))} chars')
    print('\nSources:')
    for s in final_state.get('sources', []):
        print(f'  [{s["citation_index"]}] {s["title"]}')
        print(f'      {s["url"]}')

## 5. Launch Gradio UI

Run this cell to open the full streaming Gradio interface.

In [ ]:
# Launch inline in notebook
import subprocess, sys
print('Launching Gradio UI at http://localhost:7860')
print('Or run standalone: python cartographer_gradio.py')

# Inline launch:
# exec(open('cartographer_gradio.py').read())

## 6. Next Steps

- `evals/eval_runner.ipynb` — Run the LLM-as-judge evaluation suite
- Swap LLM provider: change `CARTOGRAPHER_LLM_PROVIDER` in `.env`
- Tune the critic threshold: `CRITIC_SCORE_THRESHOLD` (default 7.0)
- Add LangSmith tracing: set `LANGCHAIN_TRACING_V2=true` in `.env`